In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from PIL import Image

# Note: You need to install groundingdino
# pip install groundingdino-py


In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Note: Grounding DINO typically requires CUDA for optimal performance


In [ ]:
try:
    from groundingdino.util.inference import load_model, load_image, predict, annotate
    print("Grounding DINO loaded successfully")
except ImportError:
    print("Grounding DINO not found. Please install with: pip install groundingdino-py")
    print("Alternative: Clone and install from https://github.com/IDEA-Research/GroundingDINO")


In [ ]:
# Initialize Grounding DINO model
# Note: You need to download the model weights first
# wget https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

try:
    CONFIG_PATH = "path/to/GroundingDINO_SwinT_OGC.py"  # Update this path
    WEIGHTS_PATH = "path/to/groundingdino_swint_ogc.pth"  # Update this path
    
    model = load_model(CONFIG_PATH, WEIGHTS_PATH)
    print("Grounding DINO model loaded successfully")
except Exception as e:
    print(f"Error loading Grounding DINO model: {e}")
    print("Please download the model weights and update the paths above")


In [ ]:
# Test data path
test_image_dir = "/Users/theo.moreau/Documents/futur/datasets/mvtec_anomaly_detection/screw/test"

# Load test image
image_path = os.path.join(test_image_dir, "good", os.listdir(os.path.join(test_image_dir, "good"))[0])

print(f"Loading image: {image_path}")

try:
    image_source, image = load_image(image_path)
    print(f"Image loaded successfully, shape: {image.shape}")
except Exception as e:
    print(f"Error loading image: {e}")
    # Fallback to PIL
    image_pil = Image.open(image_path).convert('RGB')
    image_source = np.array(image_pil)
    print(f"Image loaded with PIL, shape: {image_source.shape}")


In [ ]:
# Define text prompt for detection
TEXT_PROMPT = "screw . defect . anomaly"
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

print(f"Text prompt: {TEXT_PROMPT}")
print(f"Box threshold: {BOX_THRESHOLD}")
print(f"Text threshold: {TEXT_THRESHOLD}")

try:
    # Run prediction
    boxes, logits, phrases = predict(
        model=model,
        image=image,
        caption=TEXT_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD
    )
    
    print(f"Detected {len(boxes)} objects")
    print(f"Phrases: {phrases}")
    
    # Annotate image
    annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    
    # Display results
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Original image
    axes[0].imshow(image_source)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Annotated image
    axes[1].imshow(annotated_frame)
    axes[1].set_title(f"Grounding DINO Detection: '{TEXT_PROMPT}'")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error during prediction: {e}")
    print("Make sure Grounding DINO model is properly loaded")


In [ ]:
# Test on multiple images
def test_grounding_dino_pipeline(image_dir, text_prompt, num_images=5):
    """
    Test Grounding DINO on multiple images
    """
    good_dir = os.path.join(image_dir, "good")
    defect_dir = os.path.join(image_dir, "manipulated_front")
    
    # Test on good images
    if os.path.exists(good_dir):
        good_images = os.listdir(good_dir)[:num_images]
        for img_name in good_images:
            img_path = os.path.join(good_dir, img_name)
            try:
                image_source, image = load_image(img_path)
                boxes, logits, phrases = predict(
                    model=model,
                    image=image,
                    caption=text_prompt,
                    box_threshold=BOX_THRESHOLD,
                    text_threshold=TEXT_THRESHOLD
                )
                print(f"Good image {img_name}: {len(boxes)} objects detected - {phrases}")
            except Exception as e:
                print(f"Error processing {img_name}: {e}")
    
    # Test on defect images
    if os.path.exists(defect_dir):
        defect_images = os.listdir(defect_dir)[:num_images]
        for img_name in defect_images:
            img_path = os.path.join(defect_dir, img_name)
            try:
                image_source, image = load_image(img_path)
                boxes, logits, phrases = predict(
                    model=model,
                    image=image,
                    caption=text_prompt,
                    box_threshold=BOX_THRESHOLD,
                    text_threshold=TEXT_THRESHOLD
                )
                print(f"Defect image {img_name}: {len(boxes)} objects detected - {phrases}")
            except Exception as e:
                print(f"Error processing {img_name}: {e}")

# Run the test
try:
    test_grounding_dino_pipeline(test_image_dir, TEXT_PROMPT)
except Exception as e:
    print(f"Error in batch testing: {e}")
